In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [ ]:
"""
Test Job.get_next_request() integration with LLM_API.

This script verifies that the chat_history format returned by Job.get_next_request()
is compatible with LLM_API.process_request() method. It can be converted to a 
Jupyter notebook for interactive testing.
"""
import sys
from typing import Tuple 
import dbzero as db0
from collections import namedtuple
from statek.agent import Agent
from statek.executors.job import Job, JobDef, JobStatus
from statek.executors.chat_log_item import ChatLogItem
from statek.pyenv import PyEnv
from statek.llm_api import LLM_API, OpenRouter_API
from statek.settings import get_provider_settings
from statek.executors.utils import run_jobs_loop
from statek.future import temporal, FutureResult
from statek.executors.utils import run_agentic_loop
from statek.agents import Coordinator, Researcher

In [ ]:
# Initialize dbzero
db0.init(".dbzero_data")
db0.open("test-prefix-roon-jobs-loop")

In [ ]:
@db0.memo
class UserMessage:
    def __init__(self, username, message):
        self.username = username
        self.message = message
        db0.tags(self).add("ACTIVE_MESSAGE")

In [ ]:
def task_queue_size_func():
    return len(db0.find(UserMessage, "ACTIVE_MESSAGE"))

def check_condition(_):
    """Returns True after 2 calls, False before that."""
    active_messages = db0.find(UserMessage, "ACTIVE_MESSAGE")
    sys.stderr.write(f"CHECK CONDITION: {len(active_messages) > 0}\n");sys.stderr.flush()
    return len(active_messages) > 0

def fetch_result(future_result):
    """Fetch the result value."""
    sys.stderr.write(f"FETCH RESULT\n");sys.stderr.flush()
    active_messages = db0.find(UserMessage, "ACTIVE_MESSAGE")
    if len(active_messages) <= 0:
        raise FutureError(future_result=future_result)
    message= next(iter(active_messages))
    db0.tags(message).remove("ACTIVE_MESSAGE")
    return message.username, message.message

def send_message(message):
    sys.stderr.write(f"SENDING MESSAGE: {message}\n");sys.stderr.flush()

In [ ]:
# sample tools mocs

@temporal(complement=fetch_result, condition=check_condition)
def fetch_next_message() -> Tuple[str,str]:
    """Temporal function that returns user and message that this user send"""
    return FutureResult(
        deps=None,
        state_num=0
    )

def exit(reason: str):
    print(reason)

### Test 1: Check if run_jobs_loop can create and execute job

In [ ]:
from dotenv import load_dotenv
load_dotenv("./.env")


In [ ]:
warmup_code = """
user_and_message = fetch_next_message()
print(user_and_message[1])
"""

In [ ]:
researcher = Researcher(send_message)

In [ ]:
agent = Coordinator({"reasercher":researcher})
agent._tools.append(fetch_next_message)

In [ ]:
corutine = run_agentic_loop(agent = agent, warmup_code=warmup_code,task_queue_size_func = task_queue_size_func, max_concurrency = 2,provider = "OPENROUTER")

In [ ]:
message_1 = UserMessage("USER 1", "Tell me what is 2 + 2?")
message_2 = UserMessage("USER 2", "Jaka jutro będzie pogoda?")


In [ ]:
await corutine